# Creating AI without such libraries as Torch or TensorFlow

# Streaming batching as there is too much data

In [5]:
import numpy as np
import random
import re
from tokenizers import Tokenizer
from tokenizers.decoders import ByteLevel


file_path = "env/children_stories.txt"
tokenizer = Tokenizer.from_file("env/tokenizer.json")
tokenizer.decoder = ByteLevel()

token_buffer = np.array([], dtype=np.int32)

CHUNK_SIZE = 500_000
MIN_BUFFER_TOKENS = 500_000
MAX_BUFFER_TOKENS = 700_000


def clean_text(text):
    text = text.replace("<|endoftext|>", " endoftext ")
    text = text.replace("<|end|>", " endoftext ")
    text = re.sub(r'<\|[^|]*\|>', '', text)
    text = ' '.join(text.split())
    return text


def load_random_chunk():
    with open(file_path, "rb") as f:
        f.seek(0, 2)
        file_size = f.tell()
        pos = random.randint(0, max(1, file_size - CHUNK_SIZE))
        f.seek(pos)
        chunk = f.read(CHUNK_SIZE)

    text = chunk.decode("utf-8", errors="ignore")
    text = clean_text(text)
    tokens = tokenizer.encode(text).ids
    return np.array(tokens, dtype=np.int32)


def refill_buffer():
    global token_buffer

    while len(token_buffer) < MIN_BUFFER_TOKENS:
        new_tokens = load_random_chunk()

        if len(new_tokens) < 100:
            continue

        token_buffer = np.concatenate([token_buffer, new_tokens]) if len(token_buffer) > 0 else new_tokens

        if len(token_buffer) > MAX_BUFFER_TOKENS:
            trim_start = random.randint(0, len(token_buffer) - MAX_BUFFER_TOKENS)
            token_buffer = token_buffer[trim_start:trim_start + MAX_BUFFER_TOKENS]


def get_batch(block_size, batch_size):
    global token_buffer

    refill_buffer()

    needed = block_size * batch_size + 1
    if len(token_buffer) < needed:
        raise ValueError(
            f"Buffer too small ({len(token_buffer)}) for a batch of "
            f"batch_size={batch_size}, block_size={block_size}."
        )

    max_start = len(token_buffer) - block_size - 1

    if random.random() < 0.7:
        starts = np.random.randint(0, max_start, size=batch_size)
    else:
        max_base = max_start - batch_size * block_size
        if max_base <= 0:
            starts = np.random.randint(0, max_start, size=batch_size)
        else:
            base = random.randint(0, max_base)
            starts = [base + i * block_size for i in range(batch_size)]

    x = np.stack([token_buffer[i:i + block_size] for i in starts])
    y = np.stack([token_buffer[i + 1:i + block_size + 1] for i in starts])

    consume_up_to = int(np.max(starts)) + block_size + 1
    token_buffer = token_buffer[consume_up_to:]

    return x, y

In [ ]:
from NoTorchAI.Utils.Batch import Batch


datasets = {
    "children_stories": ("env/encoded/children_stories.bin", 1.0),
}

batch = Batch(datasets)

## Traing loop

In [2]:
import numpy as np
from tokenizers import Tokenizer
from tokenizers.decoders import ByteLevel


tokenizer = Tokenizer.from_file("env/tokenizer.json")
tokenizer.decoder = ByteLevel()

def check_model_output(model, prompt, max_tokens):
    encoded_text = tokenizer.encode(prompt).ids
    context = np.array(encoded_text, dtype=np.uint32).reshape(1, -1)

    generated = model.generate(context, max_tokens, 0.5)

    output_text = tokenizer.decode(generated[0].tolist())

    print(output_text)

In [3]:
from NoTorchAI.GlobalState.Device import Device
from NoTorchAI.GlobalState.Quant import Quant
from NoTorchAI.Gradients.Adam import Adam
from NoTorchAI.LLM.MiniGPT import MiniGPT


d_model = 384
n_heads = 8
block_layers = 7
block_size = 128

batch_size = 56
vocabulary_size = 16_000

Device("gpu")
Quant(32)

gradient = Adam(
    lr=2.2e-4,
    warmup_steps=2500,
    min_lr=1e-5,
)

model = MiniGPT(vocab_size=vocabulary_size, 
                d_model=d_model, 
                block_size=block_size,
                n_layers=block_layers,
                n_heads=n_heads,
                gradient=gradient
            )

ema_loss = None

for step in range(25_000):
    xb, yb = batch.get_batch(block_size=block_size, batch_size=batch_size)

    logits, loss = model.forward(xb, yb)

    gradient.t += 1
    model.backward()

    if ema_loss is None:
        ema_loss = loss
    else:
        ema_loss = 0.99 * ema_loss + 0.01 * loss

    if step == 1:
        print(f"step {step}, lr {gradient.get_lr():.6f}, loss {loss:.4f}, ema_loss {ema_loss:.4f}")

    if step % 100 == 0:
        check_model_output(model, "Tell me a story", 100)
        print(f"step {step}, lr {gradient.get_lr():.6f}, loss {loss:.4f}, ema_loss {ema_loss:.4f}")

    if step % 500 == 0:
        model.save("saved_model")

model.save("saved_model")

Tell me a story pointed gloomy pen pouredMPLC mag Gay Mel copsl Sirmericcalled gently pen roundmatesPD schedlamects Mon grateful artists peninkEDbie allowedachus VR championsubaurortun mine French pen Republiclam exchange Level 17 loves gentlyMikeGoogleanted streams probleGMrasunicip Dest controversGM fl Nature golden proble mortgage threw gently background couoms thiefGM gloomy putrine severPD pen vot pen rookie aersters 21 Mi conv gently thiefgy organisation� French Microsoftnes gently investment pen carries
step 0, lr 0.000010, loss 10.6697, ema_loss 10.6697
step 1, lr 0.000010, loss 10.5832, ema_loss 10.6688
Tell me a story. reporting the, to to the.. and,,... to the, a, the,burnanguage to,,, the...., a, the. spin,,, theatform and the,, and,. a. the,,.,.,. a,,,, a the,..,, the. a the, the,, a kiss a, the., the,,. a the,.
step 100, lr 0.000010, loss 7.9568, ema_loss 9.2983
Tell me a story the. to the.. Tim the the to the. the is... to.., the the a the. to.. was,. Sheensive.. He to,,

## ADD Wiki info to the model

In [ ]:
from NoTorchAI.Utils.Batch import Batch


datasets = {
    "children_stories": ("env/encoded/children_stories.bin",    0.70),
    "simple_wikipedia": ("env/encoded/simple_wikipedia.bin",     0.30),
}

batch = Batch(datasets)

In [ ]:
ema_loss = None

gradient.lr = 1e-4
gradient.min_lr = 1e-5
gradient.warmup_steps = 2000
gradient.t = 0



for step in range(15_000):

    xb, yb = batch.get_batch(block_size=block_size, batch_size=batch_size)

    logits, loss = model.forward(xb, yb)

    gradient.t += 1
    model.backward()

    if ema_loss is None:
        ema_loss = loss
    else:
        ema_loss = 0.99 * ema_loss + 0.01 * loss

    if step == gradient.warmup_steps:
        datasets = {
            "children_stories":    ("env/encoded/children_stories.bin",    0.50),
            "simple_wikipedia":    ("env/encoded/simple_wikipedia.bin",     0.50),
        }
        batch.datasets = datasets

    if step < 5 or step % 200 == 0:
        print(f"step {step}, lr {gradient.get_lr():.6f}, loss {loss:.4f}, ema_loss {ema_loss:.4f}")

    if step % 500 == 0:
        check_model_output(model, "User: Tell me a story about a dragon\nAssistant:", 120)

    if step % 1000 == 0 and step > 0:
        model.save("wiki_tuned_model")

model.save("wiki_tuned_model")

step 0, lr 0.000010, loss 1.7455, ema_loss 1.7455
User: Tell me a story about a dragon
Assistant: a dragon, and a rabbit! That way, when you are kind, you should never give up.Once upon a time, there was a little girl named Sue. Sue had a big, red ball. One day, Sue saw a big box in her yard. She wanted to see what was inside.Sue opened the box and found a toy. It was a ball. The ball was round and shiny. Sue did not know what to do. She asked her friend, Tom, to help her.Tom said, "
step 1, lr 0.000010, loss 1.7737, ema_loss 1.7458
step 2, lr 0.000010, loss 1.9462, ema_loss 1.7478
step 3, lr 0.000010, loss 1.8816, ema_loss 1.7491
step 4, lr 0.000010, loss 1.9047, ema_loss 1.7507
step 200, lr 0.000010, loss 1.7305, ema_loss 1.8395
step 400, lr 0.000020, loss 1.9522, ema_loss 1.8510
User: Tell me a story about a dragon
Assistant: a dragon, a dragon, a dragon, and a dragon. The dragon is very silly and mean. It likes to eat nuts and fish and fish.Tom and Mia are amazed. They want to be f

## Add Open Web data chuck

In [ ]:
from NoTorchAI.Utils.Batch import Batch

datasets = {
    "children_stories": ("env/encoded/children_stories.bin",    0.40),
    "simple_wikipedia": ("env/encoded/simple_wikipedia.bin",     0.35),
    "open_web":         ("env/encoded/web.bin",     0.25),
}

batch = Batch(datasets)

In [ ]:
ema_loss = None

gradient.lr = 1e-4
gradient.min_lr = 1e-5
gradient.warmup_steps = 2000
gradient.t = 0



for step in range(15_000):

    xb, yb = batch.get_batch(block_size=block_size, batch_size=batch_size)

    logits, loss = model.forward(xb, yb)

    gradient.t += 1
    model.backward()

    if ema_loss is None:
        ema_loss = loss
    else:
        ema_loss = 0.99 * ema_loss + 0.01 * loss

    if step < 5 or step % 200 == 0:
        print(f"step {step}, lr {gradient.get_lr():.6f}, loss {loss:.4f}, ema_loss {ema_loss:.4f}")

    if step % 500 == 0:
        check_model_output(model, "User: Tell me a story about a dragon\nAssistant:", 120)

    if step % 1000 == 0 and step > 0:
        model.save("wiki_tuned_model")

model.save("wiki_tuned_model")

## Hugging Face Decoder

In [4]:
from NoTorchAI.LLM.MiniGPT import MiniGPT
import numpy as np
from tokenizers import Tokenizer
from tokenizers.decoders import ByteLevel


tokenizer = Tokenizer.from_file("env/tokenizer.json")
tokenizer.decoder = ByteLevel()

# model = MiniGPT.__new__(MiniGPT)
# model: MiniGPT = model.load("saved_model")


def check_model_output(model, prompt, max_tokens):
    encoded_text = tokenizer.encode(prompt).ids
    context = np.array(encoded_text, dtype=np.uint32).reshape(1, -1)

    generated = model.generate(context, max_tokens, 0.5)

    output_text = tokenizer.decode(generated[0].tolist())

    print(output_text)


check_model_output(model, "<|user|>: Tell me a story about Yarick being sick\n<|assistant|>:", 200)

: Tell me a story about Yarick being sick
:rewurumrom^ "* the^th who*chit^ter vol other,am y^ re* spered C wh N a�ion,’le* took trionentred* otherce^ re* speOn the tINar t who,amit realinist a time us wit, other they aent st as%p whoinim* So* asG G^ re* speOnion^ re* speOnore as all T t re* speOninim*erson* C%p^ re speOnuser%p�ion woreal alion t speOnin y lookarest,al prin al commll L pres,ach* other%pseort In t whoince t re* somethingOn,amim* other* C%p^ re speOn C 1ectep shipore
